# Round 3 · Interventions at the payment moment

Detection finds accounts; it does not give anyone their money back. The platform cannot read the
conversation once it has moved to another app, but it does see the moment money moves through its
wallet, and it knows what led up to it.

**Your job:** decide, for each payment, `warn` (an interstitial the payer can click through),
`hold` (a 24-hour cool-off and a call from the safety team) or nothing. Write `out/actions_r3.csv`.

**How it is scored (in simulation):** each scam payment carries a sealed coin flip for "a warning would
have stopped it" and "a hold would have". Stopping one payment stops the rest of that victim's chain, so
early matters. Every genuine payment you hold or warn is friction, counted per payer cohort:
holds within 2%, warnings within 10%. These are simulated outcomes, not field measurements.

In [ ]:
import sys
sys.path.insert(0, "../tools")
import pandas as pd
import trailmates as tm
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

pays = pd.DataFrame(tm.payments())
pays.head()

| column | what the platform knew at that moment |
|---|---|
| `first_to_payee` | the payer has never paid this payee before |
| `payee_kind`, `payee_age_days` | a member or an external payout handle, and how long the platform has seen it |
| `payee_new_payers_7d` | how many people paid this payee for the first time in the last week |
| `payee_is_contact`, `contact_days` | the payee is someone the payer has a conversation with, and since when |
| `offplatform_contact` | of the new contacts (last 60 days) who asked the payer to move apps, the one they talk to most |
| `offplatform_contact_days`, `offplatform_contact_msgs` | how old that conversation is, and how many messages it has |
| `payee_is_offplatform_contact` | the money is going to that contact |

In [ ]:
def decide(p):
    # TODO (Round 3): the starter holds every first payment to a payee. Read who pays for that.
    # Then hold only the shape a scam has and commerce rarely does, and warn on what is merely new.
    if p.first_to_payee:
        return "hold"
    return None


acts = {p.payment_id: a for p in pays.itertuples() if (a := decide(p))}
tm.write_actions("../out/actions_r3.csv", acts)
print(pd.Series(acts).value_counts())

In [ ]:
!{sys.executable} ../tools/checkpoint.py --round 3

## Read the scoreboard

- How long after two people meet does a buyer pay a seller? How long after a new contact asks a member
  to move apps does the first payment come? Compare the distributions:

In [ ]:
first = pays[pays["first_to_payee"]]
first.groupby(first["offplatform_contact"].notna())[["contact_days", "offplatform_contact_days", "offplatform_contact_msgs", "amount"]].describe().T

- Some genuine payments will still match a scam's shape. Which ones, and is the friction worth it?
  The README names them; the debrief asks you to defend the number.

PASS? Go to `04_network.ipynb`. Stuck? `python tools/checkpoint.py --round 3 --catch-up`.